## 1) Libraries

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

## 2) Data

In [ ]:
data_raw = pd.read_csv("train.csv")
test_data_raw = pd.read_csv("test.csv")

In [ ]:
def clean_data(df):
    df = df.copy()

    # Clean percentage columns
    percent_cols = ["host_response_rate", "host_acceptance_rate"]

    for col in percent_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace("%", "", regex=False)
                .replace("nan", np.nan)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Convert yes/no columns to 1/0
    yn_cols = [
        "host_is_superhost",
        "host_has_profile_pic",
        "host_identity_verified",
        "instant_bookable",
        "has_availability"
    ]

    for col in yn_cols:
        if col in df.columns:
            df[col] = df[col].map({
                "t": 1, "f": 0,
                "yes": 1, "no": 0,
                True: 1, False: 0
            })

    # Date features
    date_cols = ["host_since", "first_review", "last_review"]

    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    reference_year = 2026

    if "host_since" in df.columns:
        df["host_years"] = reference_year - df["host_since"].dt.year
        df["host_since_missing"] = df["host_since"].isna().astype(int)

    if "first_review" in df.columns:
        df["years_since_first_review"] = reference_year - df["first_review"].dt.year
        df["first_review_missing"] = df["first_review"].isna().astype(int)

    if "last_review" in df.columns:
        df["years_since_last_review"] = reference_year - df["last_review"].dt.year
        df["last_review_missing"] = df["last_review"].isna().astype(int)

    df = df.drop(columns=[col for col in date_cols if col in df.columns])

    # Hotel-style feature
    if "property_type" in df.columns:
        prop = df["property_type"].fillna("").astype(str).str.lower()

        hotel_words = [
            "hotel",
            "boutique hotel",
            "resort",
            "hostel",
            "serviced apartment",
            "aparthotel"
        ]

        df["is_hotel"] = prop.apply(
            lambda x: int(any(word in x for word in hotel_words))
        )
    else:
        df["is_hotel"] = 0

    if "name" in df.columns:
        name_lower = df["name"].fillna("").astype(str).str.lower()
        df["name_mentions_hotel"] = name_lower.str.contains(
            "hotel|resort|inn|suite|boutique",
            regex=True
        ).astype(int)

        df["is_hotel"] = (
            (df["is_hotel"] == 1) |
            (df["name_mentions_hotel"] == 1)
        ).astype(int)

    # Amenities features
    if "amenities" in df.columns:
        amenities_text = df["amenities"].fillna("").astype(str)

        df["amenities_count"] = np.where(
            amenities_text.str.strip().eq(""),
            0,
            amenities_text.str.count(",") + 1
        )

        important_amenities = [
            "Wifi",
            "Air conditioning",
            "Kitchen",
            "Washer",
            "Dryer",
            "Free parking",
            "Pool",
            "Hot tub",
            "Gym",
            "Elevator",
            "Pets allowed",
            "Self check-in",
            "Dedicated workspace",
            "TV",
            "Heating",
            "Patio",
            "Backyard",
            "BBQ grill"
        ]

        for amenity in important_amenities:
            new_col = (
                "has_"
                + amenity.lower().replace(" ", "_").replace("-", "_")
            )

            df[new_col] = (
                amenities_text
                .str.contains(amenity, case=False, regex=False)
                .astype(int)
            )

    # Text length features
    text_cols = ["description", "host_about", "name", "neighborhood_overview"]

    for col in text_cols:
        if col in df.columns:
            df[col + "_length"] = df[col].fillna("").astype(str).str.len()
            df[col + "_word_count"] = (
                df[col]
                .fillna("")
                .astype(str)
                .str.split()
                .str.len()
            )

    # Bathroom count
    if "bathrooms_text" in df.columns:
        bath_text = df["bathrooms_text"].fillna("").astype(str).str.lower()

        df["bathrooms_num"] = (
            bath_text
            .str.extract(r"(\d+\.?\d*)")[0]
        )

        df["bathrooms_num"] = pd.to_numeric(
            df["bathrooms_num"],
            errors="coerce"
        )

        df.loc[
            bath_text.str.contains("half", na=False),
            "bathrooms_num"
        ] = 0.5

        df["shared_bath"] = bath_text.str.contains(
            "shared",
            na=False
        ).astype(int)

        df["private_bath"] = bath_text.str.contains(
            "private",
            na=False
        ).astype(int)

    # Ratio features
    if "accommodates" in df.columns and "bedrooms" in df.columns:
        df["accommodates_per_bedroom"] = (
            df["accommodates"] / (df["bedrooms"] + 1)
        )

    if "beds" in df.columns and "bedrooms" in df.columns:
        df["beds_per_bedroom"] = df["beds"] / (df["bedrooms"] + 1)

    if "bathrooms_num" in df.columns and "accommodates" in df.columns:
        df["bathrooms_per_guest"] = (
            df["bathrooms_num"] / (df["accommodates"] + 1)
        )

    if "beds" in df.columns and "accommodates" in df.columns:
        df["beds_per_guest"] = df["beds"] / (df["accommodates"] + 1)

    if "number_of_reviews" in df.columns and "host_years" in df.columns:
        df["reviews_per_host_year"] = (
            df["number_of_reviews"] / (df["host_years"] + 1)
        )

    if (
        "number_of_reviews" in df.columns
        and "years_since_first_review" in df.columns
    ):
        df["reviews_per_year_since_first_review"] = (
            df["number_of_reviews"] / (df["years_since_first_review"] + 1)
        )

    # Missingness indicators
    missing_indicator_cols = [
        "bedrooms",
        "beds",
        "bathrooms_num",
        "review_scores_rating",
        "host_response_rate",
        "host_acceptance_rate",
        "host_listings_count",
        "host_total_listings_count"
    ]

    for col in missing_indicator_cols:
        if col in df.columns:
            df[col + "_missing"] = df[col].isna().astype(int)

    df = df.replace([np.inf, -np.inf], np.nan)

    return df

In [ ]:
data_clean = clean_data(data_raw)
test_data_clean = clean_data(test_data_raw)

y = data_clean["host_is_superhost"]
test_ids = test_data_raw["id"]

X = data_clean.drop(columns=["host_is_superhost"])

if "id" in X.columns:
    X = X.drop(columns=["id"])

if "id" in test_data_clean.columns:
    X_test = test_data_clean.drop(columns=["id"])
else:
    X_test = test_data_clean.copy()

X_test = X_test.reindex(columns=X.columns)

## 3) Machine Learning Model

In [ ]:
X_full_cat = X.copy()
X_test_full_cat = X_test.copy()

cat_features = X_full_cat.select_dtypes(include=["object"]).columns.tolist()

for col in cat_features:
    X_full_cat[col] = X_full_cat[col].fillna("Missing").astype(str)
    X_test_full_cat[col] = X_test_full_cat[col].fillna("Missing").astype(str)

final_cat_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_seed=12,
    verbose=False
)

final_cat_model.fit(
    X_full_cat,
    y,
    cat_features=cat_features,
    verbose=False
)

## 4) Exporting the Predictions

In [ ]:
test_predictions = final_cat_model.predict_proba(X_test_full_cat)[:, 1]

submission = pd.DataFrame({
    "id": test_ids,
    "predicted": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()